# 01 Fra perseptron til nevrale nettverk
*Data Mining & Analytics – Samling 3, dag 1*

Géron, *Hands-On Machine Learning with Scikit-Learn and TensorFlow*, kap. 10 (s. 253–273).



In [2]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 4.5)
rng = np.random.default_rng(42)
tf.keras.utils.set_random_seed(42)
print("TensorFlow", tf.__version__)

ModuleNotFoundError: No module named 'tensorflow'

## 1. Perseptronet (Rosenblatt, 1957)

Et kunstig nevron regner ut en vektet sum av input og sender summen gjennom en aktiveringsfunksjon (her step-funksjon):

$$z = \mathbf{w}^\top \mathbf{x} + b, \qquad \hat{y} = \text{step}(z) = \begin{cases} 1 & z \ge 0 \\ 0 & z < 0 \end{cases}$$

**Læringsregelen** : for hvert treningseksempel justeres vektene i retning av feilen

$$w_i \leftarrow w_i + \eta \, (y - \hat{y}) \, x_i$$

der $\eta$ er læringsraten. Vektene endres bare når perseptronet tar feil.

In [ ]:
class Perseptron:
    """Et enkelt perseptron med step-funksjon og Rosenblatts læringsregel."""
    def __init__(self, eta=0.1, epoker=20):
        self.eta, self.epoker = eta, epoker

    def fit(self, X, y):
        self.w = np.zeros(X.shape[1])
        self.b = 0.0
        self.feil = []
        for _ in range(self.epoker):
            n_feil = 0
            for xi, yi in zip(X, y):
                oppdatering = self.eta * (yi - self.predict(xi))
                self.w += oppdatering * xi
                self.b += oppdatering
                n_feil += int(oppdatering != 0)
            self.feil.append(n_feil)
        return self

    def predict(self, X):
        return (np.dot(X, self.w) + self.b >= 0).astype(int)

In [ ]:
X_logikk = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
problemer = {"AND": np.array([0, 0, 0, 1]), "OR": np.array([0, 1, 1, 1]), "XOR": np.array([0, 1, 1, 0])}

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for a, (navn, y) in zip(ax, problemer.items()):
    p = Perseptron().fit(X_logikk, y)
    print(f"{navn}: prediksjon {p.predict(X_logikk)}  fasit {y}  feil i siste epoke: {p.feil[-1]}")
    a.scatter(X_logikk[:, 0], X_logikk[:, 1], c=y, cmap="coolwarm", s=200, edgecolor="k")
    xs = np.linspace(-0.5, 1.5, 10)
    if p.w[1] != 0:
        a.plot(xs, -(p.w[0] * xs + p.b) / p.w[1], "k--")
    a.set_xlim(-0.5, 1.5); a.set_ylim(-0.5, 1.5)
    a.set_title(navn)
plt.show()

AND og OR er lineært separerbare, én rett linje skiller klassene, og perseptronen konvergerer.
XOR er det ikke: ingen rett linje kan skille (0,1) og (1,0) fra (0,0) og (1,1). Dette påpekte Minsky og Papert i 1969,
og det bidro til at forskningen på nevrale nettverk stoppet opp i mange år 

## 2. Løsningen: flere lag (MLP)

Et flerlags perseptron har ett eller flere skjulte lag. Med en ikke-lineær aktiveringsfunksjon i det skjulte laget
kan nettverket lage en ikke-lineær beslutningsgrense.

In [ ]:
mlp = tf.keras.Sequential([
    tf.keras.Input(shape=(2,)),
    tf.keras.layers.Dense(4, activation="tanh"),     
    tf.keras.layers.Dense(1, activation="sigmoid"),   
])
mlp.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01), loss="binary_crossentropy")
mlp.fit(X_logikk, problemer["XOR"], epochs=1000, verbose=0)
print("P(y=1):", mlp.predict(X_logikk, verbose=0).ravel().round(3))

gx, gy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
gz = mlp.predict(np.c_[gx.ravel(), gy.ravel()], verbose=0).reshape(gx.shape)
plt.figure(figsize=(5, 4.5))
plt.contourf(gx, gy, gz, levels=20, cmap="coolwarm", alpha=0.6)
plt.scatter(X_logikk[:, 0], X_logikk[:, 1], c=problemer["XOR"], cmap="coolwarm", s=200, edgecolor="k")
plt.title("MLP løser XOR")
plt.colorbar(label="P(y = 1)")
plt.show()

## 3. Aktiveringsfunksjoner

Uten ikke-lineære aktiveringsfunksjoner får vi bare lineær output , uansett hvor mange lag vi bruker.
Step-funksjonen har derivert 0 nesten overalt, og kan derfor ikke trenes med "gradient decent". Dette problemet løses med sigmoid-funksjonen 
som ble introdusert av Rumelhart et al. 1986 da de introduserte backpropagation.

In [ ]:
z = np.linspace(-5, 5, 400)
sigmoid = 1 / (1 + np.exp(-z))
funksjoner = {
    "Step":       (np.where(z >= 0, 1.0, 0.0), np.zeros_like(z)),
    "Sigmoid":    (sigmoid, sigmoid * (1 - sigmoid)),
    "Tanh":       (np.tanh(z), 1 - np.tanh(z) ** 2),
    "ReLU":       (np.maximum(0, z), (z > 0).astype(float)),
    "Leaky ReLU": (np.where(z > 0, z, 0.1 * z), np.where(z > 0, 1.0, 0.1)),
}
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for navn, (f, df) in funksjoner.items():
    ax[0].plot(z, f, label=navn)
    ax[1].plot(z, df, label=navn)
ax[0].set_title("Aktiveringsfunksjon"); ax[0].set_ylim(-1.5, 2.5)
ax[1].set_title("Derivert")
for a in ax:
    a.axhline(0, color="k", lw=0.5); a.legend()
plt.show()

## 4. Et MLP på tabelldata: brystkreft

Her bruker vi et brystkreft-datasettet og sammenligner et lite MLP med logistisk regresjon.

Nevrale nettverk er følsomme for skalaen på input, så vi standardiserer (tilpasset kun på treningsdata).

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

kreft = load_breast_cancer(as_frame=True)
X = kreft.data.values
y = 1 - kreft.target.values  # omgjør 1 til ondartet

X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.2, stratify=y_dev, random_state=42)

skaler = StandardScaler().fit(X_train)
X_train_s, X_val_s, X_test_s = (skaler.transform(a) for a in (X_train, X_val, X_test))
print("Trening/validering/test:", len(X_train), len(X_val), len(X_test))

In [ ]:
modell = tf.keras.Sequential([
    tf.keras.Input(shape=(X.shape[1],)),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
modell.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
               loss="binary_crossentropy",
               metrics=[tf.keras.metrics.AUC(name="auc")])
modell.summary()

Antall parametere i et `Dense`-lag er (antall input + 1) × antall nevroner.
Antall parametre i hele nettverket blir derfor:
(30 + 1)·16 = 496, (16 + 1)·8 = 136 og (8 + 1)·1 = 9.

In [ ]:
stopp = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True)
historikk = modell.fit(X_train_s, y_train, epochs=500, batch_size=32,
                       validation_data=(X_val_s, y_val), callbacks=[stopp], verbose=0)
print("Antall epoker trent:", len(historikk.history["loss"]))

plt.plot(historikk.history["loss"], label="trening")
plt.plot(historikk.history["val_loss"], label="validering")
plt.xlabel("Epoke"); plt.ylabel("Binær kryssentropi"); plt.legend()
plt.title("Læringskurver")
plt.show()

In [ ]:
p_nn = modell.predict(X_test_s, verbose=0).ravel()
logreg = LogisticRegression(max_iter=1000).fit(X_train_s, y_train)
p_lr = logreg.predict_proba(X_test_s)[:, 1]
print(f"AUC test MLP = {roc_auc_score(y_test, p_nn):.3f}")
print(f"AUC test logistisk regresjon =  {roc_auc_score(y_test, p_lr):.3f}")

På et små tabelldatasett er nevralt nettverk sjeldent bedre enn logistisk regresjon. Dyp læring blir aktuelt når vi har mye data og fortrinnsvis data som bilder, lyd, tekst, signaler, 
der nettverket selv kan lære gode representasjoner. 